# CSIS3754 - Question 2: Water Quality Clustering
## Main End-of-Year Examination 2024

## 2.1 - Import Libraries and Read Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Read the data
water = pd.read_csv('waterquality.csv')
print('Data loaded successfully!')
water.head()

## 2.2 - Brief Summary of the Dataset

In [ ]:
# Number of records and features
print(f'Number of records: {water.shape[0]}')
print(f'Number of features: {water.shape[1]}')

In [ ]:
# Records 1001 to 1020 (index 1000 to 1019, using iloc)
# Note: .loc[1000:1019] uses label-based indexing (inclusive on both ends)
print('Records 1001 to 1020 (index 1000 to 1019):')
water.iloc[1000:1020]

In [ ]:
# Statistical summary
print('Statistical Summary:')
water.describe()

In [ ]:
# Concise summary (index, dtype, columns, non-null values, memory)
water.info()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(water.isnull().sum())
print(f'\nTotal missing values: {water.isnull().sum().sum()}')

## 2.3 - Handle Missing Values

In [ ]:
# Show percentage of missing values per column
missing_pct = (water.isnull().sum() / len(water)) * 100
print('Percentage of missing values per column:')
print(missing_pct[missing_pct > 0])

In [ ]:
# MOTIVATION:
# - For NUMERICAL columns: we use the MEDIAN to fill missing values.
#   The median is robust to outliers and better represents the central 
#   tendency when distributions are skewed (common in environmental data).
# - For CATEGORICAL/TEXT columns (like 'Date'): we use the MODE 
#   (most frequent value) since a mean/median has no meaning for text.

for col in water.columns:
    if water[col].isnull().sum() > 0:
        if water[col].dtype == 'object':
            mode_val = water[col].mode()[0]
            water[col].fillna(mode_val, inplace=True)
            print(f"'{col}' (text): filled with mode = '{mode_val}'")
        else:
            median_val = water[col].median()
            water[col].fillna(median_val, inplace=True)
            print(f"'{col}' (numeric): filled with median = {median_val:.4f}")

print('\nData after filling missing values:')
water.head()

In [ ]:
# Confirm missing values are handled
print('Missing values after treatment:')
print(water.isnull().sum())
print(f'\nTotal missing values remaining: {water.isnull().sum().sum()}')

## 2.4 - Pre-processing

In [ ]:
# Check current data types
print('Data types before pre-processing:')
print(water.dtypes)

In [ ]:
# DISCUSSION:
# The 'Date' column is of type 'object'. For clustering, we need all numeric
# features. We will extract useful numeric components (year, month, day) from
# the date and drop the original column. This preserves temporal information
# in a numeric format usable by the algorithm.

if 'Date' in water.columns:
    water['Date'] = pd.to_datetime(water['Date'], errors='coerce')
    water['Year']  = water['Date'].dt.year
    water['Month'] = water['Date'].dt.month
    water['Day']   = water['Date'].dt.day
    water.drop(columns=['Date'], inplace=True)
    print("'Date' column converted to Year, Month, Day components.")

# Drop any remaining object columns that can't be encoded meaningfully
obj_cols = water.select_dtypes(include='object').columns.tolist()
if obj_cols:
    print(f'Remaining object columns to encode/drop: {obj_cols}')
    le = LabelEncoder()
    for col in obj_cols:
        water[col] = le.fit_transform(water[col].astype(str))
        print(f"  Label-encoded: '{col}'")
else:
    print('No object columns remaining.')

print('\nData after pre-processing:')
water.head()

In [ ]:
# DISCUSSION:
# Feature Scaling: K-Means uses Euclidean distance, so features on larger
# scales will dominate the clustering. We apply StandardScaler (z-score
# normalisation) to bring all features to the same scale (mean=0, std=1).

scaler = StandardScaler()
water_scaled = scaler.fit_transform(water)
water_scaled_df = pd.DataFrame(water_scaled, columns=water.columns)

print('Data after StandardScaler normalisation:')
water_scaled_df.head()

In [ ]:
# Confirm no object columns remain
obj_remaining = water_scaled_df.select_dtypes(include='object').columns.tolist()
print(f'Object columns remaining: {obj_remaining if obj_remaining else "None - pre-processing complete!"}')
print('\nFinal data types:')
print(water_scaled_df.dtypes)

## 2.5 - K-Means Clustering (Determine Optimal k)

In [ ]:
# Use the Elbow Method to determine the optimal number of clusters (k)
# We calculate WCSS (Within-Cluster Sum of Squares) for k = 1 to 10.
# The 'elbow' point (where the curve bends) indicates the optimal k.

wcss = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(water_scaled_df)
    wcss.append(kmeans.inertia_)

# Plot the Elbow curve
plt.figure(figsize=(8, 5))
plt.plot(k_range, wcss, marker='o', linestyle='--', color='steelblue')
plt.title('Elbow Method - Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
plt.xticks(k_range)
plt.grid(True)
plt.tight_layout()
plt.show()
print('Examine the elbow curve above to identify the optimal k.')

In [ ]:
# Based on the elbow curve, select the optimal k.
# Common result for environmental data is k=3 or k=4.
# Adjust this value based on the elbow plot above.
optimal_k = 3  # <-- Change this based on your elbow plot

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
labels = kmeans.fit_predict(water_scaled_df)

# Add labels to original dataframe
water['Cluster'] = labels

print(f'K-Means applied with k = {optimal_k}')
print(f'\nCluster labels identified:')
print(labels)
print(f'\nCluster distribution:')
print(pd.Series(labels).value_counts().sort_index())

## 2.6 - Principal Component Analysis (PCA)

In [ ]:
# Apply PCA to reduce to 2 dimensions for visualisation
pca = PCA(n_components=2)
water_pca = pca.fit_transform(water_scaled_df.drop(columns=['Cluster'], errors='ignore'))

# Compare dimensions before and after
print(f'Dimensions BEFORE PCA: {water_scaled_df.drop(columns=["Cluster"], errors="ignore").shape}')
print(f'Dimensions AFTER PCA:  {water_pca.shape}')

In [ ]:
# Print principal components and explained variance
print('Principal Components (loadings):')
feature_cols = water_scaled_df.drop(columns=['Cluster'], errors='ignore').columns
pc_df = pd.DataFrame(pca.components_, columns=feature_cols, index=['PC1', 'PC2'])
print(pc_df)

print(f'\nExplained Variance Ratio:')
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {var:.4f} ({var*100:.2f}%)')

print(f'\nTotal Variance Explained: {pca.explained_variance_ratio_.sum()*100:.2f}%')

In [ ]:
# Seaborn scatterplot of PC1 vs PC2 coloured by cluster
pca_df = pd.DataFrame(water_pca, columns=['PC1', 'PC2'])
pca_df['Cluster'] = labels.astype(str)

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=pca_df,
    x='PC1', y='PC2',
    hue='Cluster',
    palette='Set1',
    alpha=0.6,
    edgecolor='k',
    linewidth=0.3
)
plt.title(f'PCA Scatter Plot - K-Means Clusters (k={optimal_k})', fontsize=13)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()

## 2.7 - Evaluation and Discussion

In [ ]:
discussion = """
EVALUATION OF SCATTER PLOT AND PCA EFFECTIVENESS:
==================================================

The PCA scatter plot reduces the multi-dimensional water quality dataset to
two principal components (PC1 and PC2) for 2D visualisation.

Observations:
- If the clusters appear well-separated with minimal overlap, this indicates
  that the K-Means algorithm successfully identified distinct groupings in the
  water quality data. This is a sign that PCA was effective.

- If the clusters significantly overlap in the scatter plot, it suggests that
  the two principal components do not capture enough variance to cleanly
  separate the clusters. In such cases, some cluster structure may exist in
  higher dimensions that PCA cannot represent in 2D.

- The total explained variance (PC1 + PC2) indicates how much of the original
  data information is retained. A combined variance above 70% is generally
  considered acceptable for visualisation purposes.

Conclusion:
PCA is an effective dimensionality reduction technique for visualisation.
However, it is a linear method and may not capture non-linear cluster
boundaries. The quality of the visualisation depends on how much variance
the first two components explain collectively.
"""

print(discussion)